# Hafta 12 · Kuantum Makine Öğrenmesi I: ML Hatırlatma ve Veri Kodlama
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab

QML bloğunun ilk haftası. Klasik bir veri satırını (özellik vektörü **x**) bir kuantum durumuna **|ψ(x)⟩** dönüştürmenin dört temel yolunu kodlayacağız: **basis**, **açı**, **yoğun açı** ve **genlik** kodlaması. Her kodlama için devreyi çizecek, `Statevector` ile elle hesabı doğrulayacak, Bloch küresinde veri bulutlarını görecek ve kodlanmış durumlar arasındaki **benzerlik matrisini** çıkaracağız (13. haftadaki kuantum çekirdeklerinin temeli).

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum ve yardımcı fonksiyonlar | 3 dk |
| A | Veri setleri (notebook içinde üretilir) | 3 dk |
| B | ML hatırlatma: özellik/etiket, eğitim/test, doğrusal model, özellik haritası | 6 dk |
| C | Basis kodlaması | 5 dk |
| D | Açı kodlaması + Bloch nokta bulutları (moons, iris) | 10 dk |
| E | Yoğun açı kodlaması | 5 dk |
| F | Genlik kodlaması (digits 0/1, PCA4 → 2 kübit) ve derinlik maliyeti | 8 dk |
| G | Önizleme: ZZ özellik haritası ve veri yeniden yükleme | 4 dk |
| H | Benzerlik matrisi \|⟨ψ(x)\|ψ(x')⟩\|² ve kodlama maliyet tablosu | 6 dk |
| I | Alıştırmalar (8 adet, `assert` ile kendini kontrol) | ödev |

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import StatePreparation, zz_feature_map
from qiskit.quantum_info import Statevector

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"

def state_to_bloch(amps):
    a, b = complex(amps[0]), complex(amps[1])
    n = np.sqrt(abs(a)**2 + abs(b)**2); a, b = a/n, b/n
    return np.array([2*(np.conj(a)*b).real, 2*(np.conj(a)*b).imag, abs(a)**2 - abs(b)**2])

def _sphere(ax, title=None):
    ax.set_box_aspect((1,1,1), zoom=1.3); ax.computed_zorder = False
    u, v = np.linspace(0, 2*np.pi, 50), np.linspace(0, np.pi, 25)
    ax.plot_surface(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)), np.outer(np.ones_like(u), np.cos(v)),
                    color="#EEF2F8", alpha=0.25, linewidth=0, shade=False)
    t = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), 0, color=GRAY, lw=0.8); ax.plot(np.cos(t), 0*t, np.sin(t), color=GRAY, lw=0.5); ax.plot(0*t, np.cos(t), np.sin(t), color=GRAY, lw=0.5)
    for d in [(1,0,0), (0,1,0), (0,0,1)]: ax.plot([-d[0], d[0]], [-d[1], d[1]], [-d[2], d[2]], color=GRAY, lw=0.7, ls="--")
    for p, s in [((0,0,1.22),"|0⟩ (z)"), ((0,0,-1.25),"|1⟩"), ((1.42,0,0),"|+⟩ (x)"), ((-1.32,0,0),"|−⟩"), ((0,1.32,0),"|+i⟩ (y)"), ((0,-1.32,0),"|−i⟩")]:
        ax.text(*p, s, ha="center", va="center", fontsize=9.5, color=NAVY)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_zlim(-1.1, 1.1); ax.view_init(elev=18, azim=30); ax.set_axis_off()
    if title: ax.set_title(title, fontsize=11, color=NAVY)

def plot_bloch(amps_list, titles=None):
    """Kübit durumlarını yan yana Bloch küresinde çizer (1. haftadaki fonksiyon)."""
    if np.ndim(amps_list) == 1: amps_list = [amps_list]
    k = len(amps_list); fig = plt.figure(figsize=(3.6*k, 3.8))
    for i, amps in enumerate(amps_list):
        ax = fig.add_subplot(1, k, i+1, projection="3d"); _sphere(ax, titles[i] if titles else None)
        x, y, z = state_to_bloch(amps)
        ax.plot([0, x], [0, y], [0, z], color=BLUE, lw=3); ax.scatter([x], [y], [z], color=BLUE, s=60, depthshade=False)
    plt.show()

def plot_bloch_cloud(clouds, y, titles=None):
    """clouds: her biri (N, 3) Bloch koordinatı dizisi olan liste; y: sınıf etiketleri (0 = mavi, 1 = turuncu)."""
    k = len(clouds); fig = plt.figure(figsize=(3.8*k, 3.9))
    for i, P in enumerate(clouds):
        ax = fig.add_subplot(1, k, i+1, projection="3d"); _sphere(ax, titles[i] if titles else None)
        for c, col in [(0, BLUE), (1, ORANGE)]:
            ax.scatter(*P[y == c].T, color=col, s=10, alpha=0.8, depthshade=False)
    plt.show()
print("hazır")

---
## A · Veri setleri
Ders boyunca kullandığımız veri setlerini **aynı tohum (seed)** ile notebook içinde üretiyoruz; dosya yüklemeye gerek yok. Tüm özellikler **[0, π]** aralığına ölçeklenmiştir (açı kodlamasına hazır).

Bu veri setlerinin CSV'leri ayrıca verilmiştir: `moons.csv`, `iris_2sinif.csv`, `digits01_pca4.csv` (sonuçlarınızı karşılaştırmak için).

In [ ]:
from sklearn.datasets import make_moons, load_iris, load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

SEED = 42

def ds_moons(n=200, noise=0.15):
    X, y = make_moons(n_samples=n, noise=noise, random_state=SEED)
    X = MinMaxScaler((0, np.pi)).fit_transform(X)            # açı kodlaması için [0, π]
    return pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y})

def ds_iris2():
    """Iris: versicolor (0) ve virginica (1) — doğrusal olarak tam ayrılamayan iki sınıf; 4 özellik."""
    d = load_iris()
    m = d.target > 0
    X = MinMaxScaler((0, np.pi)).fit_transform(d.data[m])
    y = d.target[m] - 1
    cols = ["sepal_len", "sepal_wid", "petal_len", "petal_wid"]
    df = pd.DataFrame(X, columns=cols); df["y"] = y
    return df

def ds_digits01(n_components=4):
    """sklearn digits (8x8, internet gerekmez): 0 ve 1 rakamları, PCA ile n_components özelliğe indirgenmiş."""
    d = load_digits()
    m = d.target < 2
    Xp = PCA(n_components=n_components, random_state=SEED).fit_transform(d.data[m])
    X = MinMaxScaler((0, np.pi)).fit_transform(Xp)
    df = pd.DataFrame(X, columns=[f"pc{i+1}" for i in range(n_components)]); df["y"] = d.target[m]
    return df

moons, iris, digits = ds_moons(), ds_iris2(), ds_digits01(4)
for name, df in [("moons", moons), ("iris2", iris), ("digits01_pca4", digits)]:
    print(f"{name:14s} şekil={df.shape}  sınıf sayıları={df.y.value_counts().sort_index().to_dict()}  min={df.iloc[:, :-1].values.min():.2f} max={df.iloc[:, :-1].values.max():.2f}")
moons.head()

---
## B · ML hatırlatma
- **Özellik (feature) matrisi** `X`: (örnek sayısı, özellik sayısı). **Etiket** `y`: her örneğin sınıfı.
- **Eğitim / test ayrımı:** Model eğitim verisiyle öğrenir, başarısı **görmediği** test verisiyle ölçülür.
- **Kayıp (loss):** Tahminin ne kadar yanlış olduğunu ölçen sayı; eğitim = kaybı küçültmek.
- **Doğrusal ayrılabilirlik:** Sınıfları tek bir doğru/düzlem ayırabiliyor mu?
- **Özellik haritası φ(x):** Veriyi daha yüksek boyutlu bir uzaya taşıyıp orada doğrusal ayrılabilir hâle getirmek. **Kuantum kodlama da bir özellik haritasıdır:** x → |ψ(x)⟩.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

X, y = moons[["x1", "x2"]].values, moons.y.values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print("eğitim:", Xtr.shape, " test:", Xte.shape)

clf = LogisticRegression().fit(Xtr, ytr)
print(f"Doğrusal model  eğitim doğruluğu = {clf.score(Xtr, ytr):.3f}   test doğruluğu = {clf.score(Xte, yte):.3f}")
print(f"Test log-kaybı (cross-entropy) = {log_loss(yte, clf.predict_proba(Xte)):.3f}")

xx = np.linspace(0, np.pi, 50); w, b = clf.coef_[0], clf.intercept_[0]
plt.figure(figsize=(5, 4.2))
for c, col in [(0, BLUE), (1, ORANGE)]:
    plt.scatter(*Xtr[ytr == c].T, color=col, s=15, label=f"eğitim y={c}")
    plt.scatter(*Xte[yte == c].T, facecolor="white", edgecolor=col, s=25, label=f"test y={c}")
plt.plot(xx, -(w[0]*xx + b)/w[1], "--", color=NAVY, label="doğrusal sınır")
plt.ylim(-0.1, np.pi + 0.1); plt.legend(fontsize=8); plt.title("moons: doğrusal model"); plt.show()

### Özellik haritası fikri: bir boyut eklemek ayrılabilirliği değiştirir

In [ ]:
from sklearn.datasets import make_circles
Xc, yc = make_circles(200, noise=0.08, factor=0.45, random_state=SEED)
lin = LogisticRegression().fit(Xc, yc).score(Xc, yc)
phi = np.c_[Xc, (Xc**2).sum(1)]                 # φ(x) = (x1, x2, x1² + x2²)
lin_phi = LogisticRegression().fit(phi, yc).score(phi, yc)
print(f"Ham 2B özelliklerle doğrusal model doğruluğu : {lin:.2f}")
print(f"φ(x) ile 3B'de doğrusal model doğruluğu      : {lin_phi:.2f}")
print("Kuantum kodlama da bir φ: x → |ψ(x)⟩ (2ⁿ boyutlu karmaşık vektör).")

---
## C · Basis kodlaması
Bir bit dizisi doğrudan hesaplama baz durumuna yazılır: `'101'` → |101⟩. Gereken tek kapı **X**. Maliyet: bit başına 1 kübit, derinlik 1.

Bit sırası: **Qiskit sırası** (stringin en sağındaki karakter q₀).

In [ ]:
def encode_basis(bits):
    """'101' gibi bir bit dizisini |101⟩ durumuna kodlar (en sağdaki bit -> q0)."""
    n = len(bits)
    qc = QuantumCircuit(n, name="basis")
    for i, b in enumerate(reversed(bits)):
        if b == "1":
            qc.x(i)
    return qc

qc = encode_basis("101")
display(qc.draw("mpl"))
sv = Statevector(qc)
print("Durum vektörü:", sv.data.real)
print("Sözlük gösterimi:", {str(k): complex(v).real for k, v in sv.to_dict().items()}, " -> indeks", int("101", 2))

Bir **veri kümesinin tamamı** da tek bir durumda eşit süperpozisyon olarak tutulabilir (ör. {001, 011, 110}). Bu durumun hazırlanması genel olarak pahalıdır; burada `StatePreparation` ile kuruyoruz.

In [ ]:
data = ["001", "011", "110"]
amps = np.zeros(8); amps[[int(b, 2) for b in data]] = 1
amps /= np.linalg.norm(amps)
qc = QuantumCircuit(3); qc.append(StatePreparation(amps), range(3))
print({str(k): round(float(v), 3) for k, v in Statevector(qc).probabilities_dict().items() if v > 1e-9})

---
## D · Açı kodlaması (angle encoding)
Her özellik bir kübitin **Ry dönüş açısı** olur:

$$R_y(x)|0\rangle = \begin{bmatrix}\cos(x/2)\\ \sin(x/2)\end{bmatrix}, \qquad |\psi(x)\rangle = R_y(x_{d-1})|0\rangle \otimes \cdots \otimes R_y(x_0)|0\rangle$$

- d özellik → **d kübit**, derinlik **1**, **CNOT yok** (çarpım durumu, dolanıklık yok).
- Bloch küresinde: x = 0 → |0⟩ (kuzey kutbu), x = π/2 → |+⟩, x = π → |1⟩ (güney kutbu). Nokta xz meridyeni üzerinde kalır.

In [ ]:
def encode_angle(x):
    """Her özellik bir kübitte Ry(x_i)|0⟩. x_i -> q_i."""
    x = np.asarray(x, dtype=float)
    qc = QuantumCircuit(len(x), name="angle")
    for i, xi in enumerate(x):
        qc.ry(xi, i)
    return qc

x = np.array([np.pi/2, np.pi/3])
qc = encode_angle(x)
display(qc.draw("mpl"))

# Elle hesap: q1 ⊗ q0 (Qiskit sırası)
q0 = np.array([np.cos(x[0]/2), np.sin(x[0]/2)])
q1 = np.array([np.cos(x[1]/2), np.sin(x[1]/2)])
elle = np.kron(q1, q0)
print("Elle   :", elle)
print("Qiskit :", Statevector(qc).data.real)
print("Eşit mi?", np.allclose(elle, Statevector(qc).data))

In [ ]:
# Tek kübitte Ry(x) Bloch üzerinde tek tek
xs = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]
plot_bloch([[np.cos(v/2), np.sin(v/2)] for v in xs], ["x = 0", "x = π/4", "x = π/2", "x = 3π/4", "x = π"])

### Moons verisi: her özellik kendi kübitinde (2 kübit)
Açı kodlamasında her kübitin Bloch noktası yalnızca kendi özelliğine bağlıdır: `(sin x, 0, cos x)`.

In [ ]:
def bloch_angle(xcol):
    xcol = np.asarray(xcol)
    return np.c_[np.sin(xcol), 0*xcol, np.cos(xcol)]

# Formülü bir örnek üzerinde Statevector ile kontrol edelim
x0 = X[0, 0]
print("formül :", bloch_angle([x0])[0], "\nqiskit :", state_to_bloch(Statevector(encode_angle([x0])).data))

plot_bloch_cloud([bloch_angle(X[:, 0]), bloch_angle(X[:, 1])], y, ["q0 ← Ry(x1)", "q1 ← Ry(x2)"])

### Iris: 4 özellik → 4 kübit

In [ ]:
Xi, yi = iris.iloc[:, :4].values, iris.y.values
plot_bloch_cloud([bloch_angle(Xi[:, k]) for k in range(4)], yi, [f"q{k} ← {c}" for k, c in enumerate(iris.columns[:4])])
qc = encode_angle(Xi[0]); display(qc.draw("mpl"))
print("Kübit:", qc.num_qubits, " derinlik:", qc.depth(), " CNOT:", qc.count_ops().get("cx", 0))

---
## E · Yoğun açı kodlaması (dense angle encoding)
Bir kübitin Bloch küresinde **iki** serbest açısı vardır (θ, φ). İkisini de kullanırsak kübit başına **2 özellik** sığar:

$$|\psi\rangle = R_z(x_2)\,R_y(x_1)|0\rangle = \begin{bmatrix} e^{-i x_2/2}\cos(x_1/2) \\ e^{i x_2/2}\sin(x_1/2)\end{bmatrix}$$

Bloch koordinatı: (sin x₁ cos x₂, sin x₁ sin x₂, cos x₁). Kübit sayısı yarıya iner, derinlik 2.

In [ ]:
def encode_dense_angle(x):
    """Kübit k: Ry(x[2k]) ardından Rz(x[2k+1]). Tek sayıda özellik varsa son Rz atlanır."""
    x = np.asarray(x, dtype=float)
    n = int(np.ceil(len(x) / 2))
    qc = QuantumCircuit(n, name="dense")
    for k in range(n):
        qc.ry(x[2*k], k)
        if 2*k + 1 < len(x):
            qc.rz(x[2*k + 1], k)
    return qc

x = np.array([np.pi/2, np.pi/2])
qc = encode_dense_angle(x); display(qc.draw("mpl"))
sv = Statevector(qc).data
print("Durum:", sv, "\nBloch:", state_to_bloch(sv), " -> |+i⟩ (y ekseni)")

def bloch_dense(t, p):
    return np.c_[np.sin(t)*np.cos(p), np.sin(t)*np.sin(p), np.cos(t)]
plot_bloch_cloud([bloch_dense(X[:, 0], X[:, 1]), bloch_dense(X[:, 0], 2*X[:, 1])], y,
                 ["θ = x1, φ = x2 (tek kübit)", "θ = x1, φ = 2·x2 (tüm küre)"])

---
## F · Genlik kodlaması (amplitude encoding)
2ⁿ özellik doğrudan n kübitin **genlikleri** olur: |ψ(x)⟩ = Σ (xᵢ/‖x‖) |i⟩.
1. Uzunluğu 2'nin kuvvetine **sıfırla doldur** (padding).
2. **Normalize et** (‖x‖ = 1).
3. Qiskit: `StatePreparation(vektör)` (ya da `qc.initialize(...)`, ki bu önce sıfırlama ekler).

⚠️ Kübit sayısı logaritmik (çok az) ama **devre derinliği üstel** büyür: genel bir vektör için ~2ⁿ CNOT.

In [ ]:
def encode_amplitude(x):
    """x'i 2'nin kuvvetine sıfırla doldurur, normalize eder ve StatePreparation ile yükler."""
    x = np.asarray(x, dtype=float)
    n = max(1, int(np.ceil(np.log2(len(x)))))
    v = np.zeros(2**n); v[:len(x)] = x
    v = v / np.linalg.norm(v)
    qc = QuantumCircuit(n, name="amplitude")
    qc.append(StatePreparation(v), range(n))
    return qc

x = np.array([4, 2, 1, 2])            # ‖x‖ = 5
qc = encode_amplitude(x); display(qc.draw("mpl"))
print("Beklenen genlikler x/‖x‖:", x / 5)
print("Qiskit:", Statevector(qc).data.real)
print("Olasılıklar:", Statevector(qc).probabilities())

In [ ]:
# İçeride ne var? Ayrıştırılmış devre
t = transpile(qc, basis_gates=["cx", "u"], optimization_level=1)
display(t.draw("mpl"))
print("Derinlik:", t.depth(), " CNOT:", t.count_ops().get("cx", 0))

### Digits 0/1 (PCA ile 64 → 4 özellik) → 2 kübit
8×8 = 64 piksellik görüntüleri PCA ile 4 bileşene indirip 4 genliğe (2 kübit) yüklüyoruz.

In [ ]:
Xd, yd = digits.iloc[:, :4].values, digits.y.values
A = Xd / np.linalg.norm(Xd, axis=1, keepdims=True)       # her satır bir durum vektörü
i0, i1 = np.where(yd == 0)[0][0], np.where(yd == 1)[0][0]
for i in [i0, i1]:
    sv = Statevector(encode_amplitude(Xd[i])).data.real
    print(f"rakam {yd[i]}: x = {Xd[i]}  ->  genlikler {sv}  (elle: {A[i]})")

fig, axs = plt.subplots(1, 2, figsize=(9, 3))
for ax, i, col in [(axs[0], i0, BLUE), (axs[1], i1, ORANGE)]:
    ax.bar(["|00⟩", "|01⟩", "|10⟩", "|11⟩"], A[i], color=col); ax.set_ylim(0, 1); ax.set_title(f"rakam {yd[i]}")
plt.tight_layout(); plt.show()

# PCA ne kadar bilgi tutuyor?
d = load_digits(); m = d.target < 2
cum = np.cumsum(PCA(8).fit(d.data[m]).explained_variance_ratio_)
print("Açıklanan varyans: 4 bileşen = %.2f, 8 bileşen = %.2f" % (cum[3], cum[7]))

### Genlik kodlamasının derinlik maliyeti

In [ ]:
rows = []
rng = np.random.default_rng(0)
for n in range(1, 7):
    v = rng.random(2**n)
    t = transpile(encode_amplitude(v), basis_gates=["cx", "u"], optimization_level=1)
    rows.append({"kübit n": n, "özellik 2ⁿ": 2**n, "derinlik": t.depth(), "CNOT": t.count_ops().get("cx", 0)})
pd.DataFrame(rows)

⚠️ **Ölçek kaybı:** Normalizasyon vektörün uzunluğunu siler. `x` ve `2x` aynı duruma gider:

In [ ]:
a = Statevector(encode_amplitude([4, 2, 1, 2])); b = Statevector(encode_amplitude([8, 4, 2, 4]))
print("x ve 2x aynı durum mu?", np.allclose(a.data, b.data))

---
## G · Önizleme: ZZ özellik haritası ve veri yeniden yükleme
- **ZZ (IQP tipi) özellik haritası:** H katmanı + özelliklere bağlı faz kapıları + özellik **çarpımlarına** bağlı ZZ etkileşimleri. Dolanık durum üretir; klasik olarak simüle edilmesi zor olduğu düşünülen bir aileye aittir. 13. haftada QSVM'de kullanacağız.
- **Veri yeniden yükleme (data re-uploading):** x'i devreye tek sefer değil, eğitilebilir kapılarla araya girerek **L kez** yüklemek. Model daha zengin fonksiyonlar (cos(x), cos(2x), … cos(Lx)) öğrenebilir. 14. haftada kullanacağız.

In [ ]:
fm = zz_feature_map(2, reps=1)
display(fm.decompose().draw("mpl"))
sv = Statevector(fm.assign_parameters([1.0, 0.5]))
print("x = (1.0, 0.5) için ZZ durumu:", np.round(sv.data, 3))
print("Olasılıklar:", np.round(sv.probabilities(), 3), " -> hepsi eşit: bilgi tamamen FAZLARDA (reps=1)")
rank = np.linalg.matrix_rank(sv.data.reshape(2, 2), tol=1e-9)
print("2×2 rank:", rank, "->", "dolanık" if rank == 2 else "çarpım durumu", "(açı kodlaması ise her zaman rank 1)")

In [ ]:
def reupload_expz(x, thetas):
    """Tek kübit: L kez [Ry(x) -> Rz(θ) -> Ry(θ')]; çıktı ⟨Z⟩."""
    L = len(thetas) // 2
    qc = QuantumCircuit(1)
    for l in range(L):
        qc.ry(x, 0); qc.rz(thetas[2*l], 0); qc.ry(thetas[2*l + 1], 0)
    p = Statevector(qc).probabilities()
    return p[0] - p[1]

xs = np.linspace(-np.pi, np.pi, 120); r = np.random.default_rng(3)
plt.figure(figsize=(9, 3))
for L in [1, 2, 4]:
    th = r.uniform(0, 2*np.pi, 2*L)
    plt.plot(xs, [reupload_expz(v, th) for v in xs], label=f"L = {L}")
plt.legend(); plt.xlabel("x"); plt.ylabel("⟨Z⟩"); plt.title("Veri yeniden yükleme: katman arttıkça daha karmaşık f(x)"); plt.show()

---
## H · Benzerlik matrisi ve maliyet tablosu
İki kodlanmış durum arasındaki benzerlik (sadakat, *fidelity*):
$$k(x, x') = |\langle \psi(x) | \psi(x') \rangle|^2 \in [0, 1]$$
Açı kodlamasında bu formül kapalı biçimde yazılabilir: $\prod_i \cos^2\big((x_i - x'_i)/2\big)$.
Veriyi **sınıfa göre sıralayıp** matrisi çizersek, iyi bir kodlamada köşegen bloklar (aynı sınıf) koyu, köşegen dışı (farklı sınıf) açık görünür. **Gelecek hafta bu matris doğrudan bir SVM'e verilecek (QSVM).**

In [ ]:
def statevector_of(qc):
    return Statevector(qc).data

def gram(Xs, encoder):
    S = np.array([statevector_of(encoder(x)) for x in Xs])
    return np.abs(S.conj() @ S.T)**2

def show_gram(K, n0, title):
    plt.imshow(K, cmap="Blues", vmin=0, vmax=1); plt.colorbar(fraction=0.046)
    plt.axhline(n0 - 0.5, color=ORANGE); plt.axvline(n0 - 0.5, color=ORANGE); plt.title(title, fontsize=10)

r = np.random.default_rng(7)
sel = np.r_[r.choice(np.where(y == 0)[0], 25, replace=False), r.choice(np.where(y == 1)[0], 25, replace=False)]
seld = np.r_[r.choice(np.where(yd == 0)[0], 25, replace=False), r.choice(np.where(yd == 1)[0], 25, replace=False)]
K1, K2 = gram(X[sel], encode_angle), gram(X[sel], encode_dense_angle)
K3, K4 = gram(Xd[seld], encode_angle), gram(Xd[seld], encode_amplitude)

plt.figure(figsize=(16, 3.8))
for i, (K, t) in enumerate([(K1, "moons · açı (2 kübit)"), (K2, "moons · yoğun açı (1 kübit)"), (K3, "digits · açı (4 kübit)"), (K4, "digits · genlik (2 kübit)")]):
    plt.subplot(1, 4, i+1); show_gram(K, 25, t)
plt.tight_layout(); plt.show()

def within_between(K, n0=25):
    w = np.r_[K[:n0, :n0].ravel(), K[n0:, n0:].ravel()].mean(); b = K[:n0, n0:].mean()
    return w, b
for K, t in [(K1, "moons açı"), (K2, "moons yoğun"), (K3, "digits açı"), (K4, "digits genlik")]:
    w, b = within_between(K); print(f"{t:14s} sınıf içi ort. = {w:.2f}   sınıflar arası ort. = {b:.2f}   fark = {w - b:.2f}")

In [ ]:
# Açı kodlaması için kapalı formül ile doğrulama
xa, xb = X[sel[0]], X[sel[1]]
formul = np.prod(np.cos((xa - xb)/2)**2)
print("Statevector ile:", K1[0, 1], "  formül ile:", formul)

### Kodlama maliyet tablosu (d = 4 özellikli bir örnek için, `transpile` sonrası)

In [ ]:
x4 = Xd[0]
encoders = {"basis (4 özellik × 3 bit)": lambda x: encode_basis("".join(format(int(v / np.pi * 7), "03b") for v in x)),
            "açı": encode_angle, "yoğun açı": encode_dense_angle, "genlik": encode_amplitude,
            "ZZ (reps=2)": lambda x: zz_feature_map(len(x), reps=2).assign_parameters(x)}
rows = []   # not: yoğun açıda Ry+Rz, transpile sonrası tek bir U kapısına birleşir (derinlik 1)
for name, enc in encoders.items():
    t = transpile(enc(x4), basis_gates=["cx", "u"], optimization_level=1)
    rows.append({"kodlama": name, "kübit": t.num_qubits, "derinlik": t.depth(), "CNOT": t.count_ops().get("cx", 0)})
pd.DataFrame(rows)

---
## I · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · Açı kodlamasını NumPy ile yazın
`angle_state(x)` fonksiyonu Qiskit kullanmadan, `np.kron` ile açı kodlanmış durum vektörünü döndürsün (Qiskit sırası: en yüksek numaralı kübit kron zincirinin en solunda).

In [ ]:
def angle_state(x):
    # TODO
    pass

for x in [[0.3], [np.pi/2, np.pi/3], iris.iloc[5, :4].values]:
    assert np.allclose(angle_state(x), Statevector(encode_angle(x)).data)
print("Alıştırma 1 ✓")

### Alıştırma 2 · Tamsayıyı basis kodlaması
`encode_int(k, n)`: 0 ≤ k < 2ⁿ tamsayısını n kübitlik |k⟩ durumuna kodlayan devreyi döndürsün (ipucu: `format(k, f"0{n}b")` + `encode_basis`).

In [ ]:
def encode_int(k, n):
    # TODO
    pass

for k, n in [(5, 3), (0, 2), (13, 4)]:
    sv = Statevector(encode_int(k, n)).data
    assert np.isclose(abs(sv[k]), 1.0)
assert encode_int(6, 3).count_ops().get("x", 0) == 2
print("Alıştırma 2 ✓")

### Alıştırma 3 · Yoğun açı kodlamasının Bloch koordinatları
`dense_bloch(t, p)` (Rz(p)·Ry(t)|0⟩ durumunun Bloch vektörü) fonksiyonunu formülle yazın ve Qiskit ile doğrulayın.

In [ ]:
def dense_bloch(t, p):
    # TODO: (x, y, z) döndürün
    pass

for t, p in [(np.pi/2, np.pi/2), (1.0, 2.0), (np.pi, 0.3), (0.2, 3.0)]:
    ref = state_to_bloch(Statevector(encode_dense_angle([t, p])).data)
    assert np.allclose(dense_bloch(t, p), ref)
print("Alıştırma 3 ✓")

### Alıştırma 4 · Genlik kodlaması için ön işleme
`amp_prepare(x)`: vektörü 2'nin kuvvetine sıfırla doldurup normalize etsin ve `(vektör, kübit_sayısı)` döndürsün. Sıfır vektör için `ValueError` fırlatsın.

In [ ]:
def amp_prepare(x):
    # TODO
    pass

v, n = amp_prepare([3, 4]);        assert n == 1 and np.allclose(v, [0.6, 0.8])
v, n = amp_prepare([1, 1, 1]);     assert n == 2 and np.allclose(v, [1, 1, 1, 0] / np.sqrt(3))
v, n = amp_prepare(np.ones(64));   assert n == 6 and np.isclose(np.linalg.norm(v), 1)
try:
    amp_prepare([0, 0]); assert False, "ValueError bekleniyordu"
except ValueError:
    pass
print("Alıştırma 4 ✓")

### Alıştırma 5 · Sadakat (fidelity) ve açı kodlaması formülü
`fidelity(a, b)` = |⟨a|b⟩|². Sonra açı kodlaması için kapalı formülü `angle_kernel(x, xp)` = ∏ cos²((xᵢ − x'ᵢ)/2) yazın ve ikisinin eşit olduğunu gösterin.

In [ ]:
def fidelity(a, b):
    # TODO
    pass

def angle_kernel(x, xp):
    # TODO
    pass

for i, j in [(0, 1), (3, 40), (10, 99)]:
    a, b = Statevector(encode_angle(Xi[i])).data, Statevector(encode_angle(Xi[j])).data
    assert np.isclose(fidelity(a, b), angle_kernel(Xi[i], Xi[j]))
assert np.isclose(fidelity([1, 0], [0, 1]), 0) and np.isclose(fidelity([1, 0], [1j, 0]), 1)
print("Alıştırma 5 ✓")

### Alıştırma 6 · Benzerlik matrisinin özellikleri
`similarity_matrix(Xs, encoder)` fonksiyonunu yazın. Sonuç **simetrik** olmalı, **köşegeni 1** olmalı ve **pozitif yarı-tanımlı** (tüm özdeğerler ≥ 0) olmalı. Bu üç özellik, matrisin gelecek hafta SVM'e çekirdek olarak verilebilmesi için gereklidir.

In [ ]:
def similarity_matrix(Xs, encoder):
    # TODO
    pass

K = similarity_matrix(Xi[:30], encode_dense_angle)
assert K.shape == (30, 30)
assert np.allclose(K, K.T) and np.allclose(np.diag(K), 1)
assert np.linalg.eigvalsh(K).min() > -1e-9
print("Alıştırma 6 ✓  en küçük özdeğer:", np.linalg.eigvalsh(K).min())

### Alıştırma 7 · Ham veriyi açıya ölçekleme
`to_angles(Xtrain, Xtest)`: eğitim verisinin sütun min/max değerleriyle hem eğitimi hem testi [0, π]'ye ölçeklesin (test için **eğitimin** min/max'ı kullanılır — veri sızıntısı olmasın). Test değerleri aralık dışına taşarsa `np.clip` ile [0, π]'ye kırpın.

In [ ]:
def to_angles(Xtrain, Xtest):
    # TODO: (Atrain, Atest) döndürün
    pass

raw = load_iris().data[50:]                  # ölçeklenmemiş iris (cm)
Rtr, Rte = raw[:70], raw[70:]
Atr, Ate = to_angles(Rtr, Rte)
assert np.allclose(Atr.min(0), 0) and np.allclose(Atr.max(0), np.pi)
assert Ate.min() >= 0 and Ate.max() <= np.pi
lo, hi = Rtr.min(0), Rtr.max(0)
assert np.allclose(Ate[0], np.clip((Rte[0] - lo) / (hi - lo) * np.pi, 0, np.pi))
print("Alıştırma 7 ✓")

### Alıştırma 8 · Benzerlikle sınıflandırma (13. haftaya köprü)
`predict_by_similarity(Xtr, ytr, Xte, encoder)`: her test örneği için eğitimdeki **sınıf 0** örneklerine ortalama benzerliği ve **sınıf 1** örneklerine ortalama benzerliği hesaplayıp büyük olan sınıfı tahmin etsin. Moons verisinde açı kodlamasıyla test doğruluğu **≥ 0.85** olmalı.

In [ ]:
def predict_by_similarity(Xtr, ytr, Xte, encoder):
    # TODO
    pass

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
pred = predict_by_similarity(Xtr, ytr, Xte, encode_angle)
acc = (pred == yte).mean()
print("test doğruluğu:", acc)
assert acc >= 0.85
print("Alıştırma 8 ✓")

---
### Haftanın özeti
- QML bloğunda odak **CQ**: klasik veri + kuantum model. İlk adım her zaman **kodlama** U(x): x → |ψ(x)⟩.
- **Basis:** bit başına 1 kübit, sadece X kapıları; tamsayı/kategorik veri için.
- **Açı:** özellik başına 1 kübit, `Ry(x)`; derinlik 1, dolanıklık yok. Özellikler **[0, π]** aralığına ölçeklenir.
- **Yoğun açı:** `Ry + Rz`, kübit başına 2 özellik.
- **Genlik:** 2ⁿ özellik → n kübit; normalizasyon gerekir (ölçek kaybolur); derinlik ~2ⁿ → **veri yükleme darboğazı**.
- ZZ özellik haritası ve veri yeniden yükleme: daha ifade gücü yüksek kodlamalar (13–14. haftalar).
- Benzerlik matrisi |⟨ψ(x)|ψ(x')⟩|² kodlamanın sınıfları ne kadar ayırdığını gösterir.

**Gelecek hafta:** Kuantum çekirdekler ve QSVM — bugünkü benzerlik matrisini doğrudan bir SVM'e vereceğiz; `FidelityQuantumKernel`, ZZ özellik haritası ve klasik RBF çekirdeğiyle karşılaştırma.